# m49 lab · Fluoroscopy and radiation protection

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision.*

You compute how X-rays are attenuated by tissue, bone and a platinum electrode; project a chest phantom into a
fluoroscopy image; add quantum noise and see why a catheter is easy to see and the heart is not; trade pulse rate
against dose and motion; check that the kerma-area product does not depend on distance; budget staff dose; and
compute reference levels from synthetic dose logs the two ways national surveys do.

| Part | What you do | Data |
| --- | --- | --- |
| A | Attenuation and half-value layers (eq-m49-beer) | NIST attenuation coefficients |
| B | A projection image of a chest phantom, exact and by ray marching | our own phantom |
| C | Quantum noise and the dose needed to see a structure (eq-m49-contrast, eq-m49-cnr) | our own phantom |
| D | Pulse rate: dose against motion | illustrative motion |
| E | Distance, kerma-area product and staff dose (eq-m49-kap, eq-m49-staff) | illustrative settings |
| F | Reference levels from dose logs | synthetic logs, published levels for comparison |

Attenuation coefficients are NIST values at a single photon energy of 60 keV (Hubbell and Seltzer, NIST SRD 126);
real beams are polyenergetic. Phantom sizes, photon counts, pulse rates, staff positions and dose-log parameters
are illustrative. Each part ends with **checks**; the lab runs in well under a minute and needs no downloads.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/raghavendrasri/open-ep-mapping-lab",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

from epml import fluoro as fl

## Part A · Attenuation and half-value layers

A narrow beam of photons of one energy loses a fixed fraction per millimetre: $I = I_0 e^{-\mu x}$ (eq-m49-beer).
The linear attenuation coefficient $\mu$ is the mass attenuation coefficient from the NIST tables times density.
The half-value layer, $\ln 2/\mu$, is the thickness that halves the beam.

In [ ]:
rows = []
for m in fl.MATERIALS:
    mu = fl.mu_per_mm(m)
    rows.append((m, mu, fl.half_value_layer(mu)))
    print(f"{m:14s} mu = {mu:.5f} /mm   half-value layer = {fl.half_value_layer(mu):7.3f} mm")
body = 220.0   # front-to-back path through the phantom's body, mm (illustrative)
t_body = fl.transmission(fl.mu_per_mm("soft tissue"), body)
print(f"Transmission through {body:.0f} mm of soft tissue: {t_body:.4f} (1 photon in {1 / t_body:.0f})")
x = np.linspace(0, 100, 201)
for m in ("soft tissue", "cortical bone"):
    plt.plot(x, fl.transmission(fl.mu_per_mm(m), x), label=m)
plt.xlabel("thickness (mm)"); plt.ylabel("transmitted fraction"); plt.legend(); plt.show()

In [ ]:
# Checks
for m, mu, hvl in rows:
    assert abs(fl.transmission(mu, hvl) - 0.5) < 1e-12
hvl = {m: h for m, _, h in rows}
assert hvl["platinum"] < 0.1 < hvl["cortical bone"] < hvl["soft tissue"]
assert 1 / t_body > 100

A platinum electrode a fraction of a millimetre thick stops half the beam; it takes about 3 cm of soft tissue to
do the same. Most photons never reach the detector: the patient absorbs them, which is the dose.

## Part B · A projection image of a chest phantom

The phantom is a set of ellipsoids: a soft-tissue body, a blood-filled heart, a bone spine and four platinum ring
electrodes of a catheter inside the heart. For a beam parallel to $y$, the line integral of $\mu$ through an
ellipsoid is exact (a chord length times its attenuation, nested objects adding their difference). A numerical
march through the phantom checks it.

In [ ]:
phantom = fl.chest_phantom()
xs = np.arange(-90, 110, 1.0) + 0.5
zs = np.arange(-80, 70, 1.0) + 0.5
X, Z = np.meshgrid(xs, zs)
L = fl.line_integral(phantom, X, Z)
probe_x = np.r_[20.0, 45.0, 0.0, -60.0]; probe_z = np.r_[0.0, 40.0, -60.0, 20.0]
L_march = fl.line_integral_voxels(phantom, probe_x, probe_z, step=0.02)
gap = np.max(np.abs(fl.line_integral(phantom, probe_x, probe_z) - L_march))
plt.figure(figsize=(6, 4.5))
plt.imshow(np.clip(L, 4.5, 7.5)[::-1], cmap="gray_r", extent=[xs[0], xs[-1], zs[0], zs[-1]])
plt.xlabel("x (mm)"); plt.ylabel("z, head up (mm)"); plt.title("line integral of mu (darker = more attenuation)")
plt.colorbar(label="line integral"); plt.show()
print(f"Largest difference between exact and marched line integrals at the probes: {gap:.4f}")

In [ ]:
# Checks
assert gap < 0.02
assert L.max() > 20            # through an electrode
assert np.ptp(L[np.abs(X - 45) < 1]) < 2   # a column through the heart only varies with the body's shape

## Part C · Quantum noise, and what it takes to see a structure

Photons arrive at random: a pixel expecting $N$ photons sees a Poisson count with standard deviation $\sqrt N$. The
subject contrast of an object is $C = 1 - e^{-\Delta\mu\, t}$ (eq-m49-contrast), and with Poisson noise the
contrast-to-noise ratio is $C\sqrt N$ (eq-m49-cnr). Here: blood against soft tissue across the heart, a
platinum electrode in blood, and bone against tissue across the spine.

In [ ]:
def contrast_at(name, x, z):
    part = next(p for p in phantom if p.name == name)
    return 1 - np.exp(-part.delta_mu() * part.chord(x, z)), np.exp(-(fl.line_integral(phantom, x, z) - part.delta_mu() * part.chord(x, z)))

targets = {"heart": ("heart", 45.0, 40.0), "electrode": ("electrode 3", 20.0, 0.0), "spine": ("spine", 0.0, -60.0)}
need = {}
print("structure   contrast    photons per pixel (incident) for CNR 3")
for label, (name, x, z) in targets.items():
    C, t_bg = contrast_at(name, x, z)
    need[label] = fl.photons_for_cnr(C, 3) / t_bg     # photons reaching the detector, divided by transmission
    print(f"{label:10s} {C:9.4%}   {need[label]:14,.0f}")
rng = np.random.default_rng(3)
fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, n in zip(axes, (1e3, 1e4, 1e6)):
    counts = fl.projection(phantom, X, Z, n, rng)
    ax.imshow(np.clip(-np.log(np.maximum(counts, 0.5) / n), 4.5, 7.5)[::-1], cmap="gray_r")
    ax.set_title(f"{n:,.0f} photons per pixel"); ax.axis("off")
plt.show()
ratio = need["heart"] / need["electrode"]
print(f"Seeing the heart's blood against tissue needs {ratio:,.0f} times the dose needed to see an electrode.")

In [ ]:
# Checks
assert need["electrode"] < need["spine"] < need["heart"]
assert ratio > 1e4
# simulated CNR agrees with eq-m49-cnr
C, t_bg = contrast_at("spine", 0.0, -60.0)
n = 2000.0
bg = rng.poisson(n * t_bg, 100_000); obj = rng.poisson(n * t_bg * (1 - C), 100_000)
assert abs((bg.mean() - obj.mean()) / bg.std() - fl.cnr(C, n * t_bg)) / fl.cnr(C, n * t_bg) < 0.03

Blood and muscle attenuate X-rays almost equally, so fluoroscopy shows the catheter, the bones and (in a real
chest) the air-filled lungs, but not the inside of the heart. That is why mapping systems, intracardiac echo and
image fusion carry so much of the anatomical work in EP (m50, m51, m55). The phantom has no lungs; air around the
heart would add contrast at its border that blood and muscle alone do not give.

## Part D · Pulse rate: dose against motion

Pulsed fluoroscopy fires short pulses; at a fixed dose per pulse, dose rate is proportional to the pulse rate. Fewer
pulses mean less dose, but a moving catheter is shown where it was at the last pulse. Here a tip moves 5 mm back
and forth at 1.2 cycles per second (illustrative).

In [ ]:
rates = np.array([30.0, 15.0, 7.5, 3.75])
dose_rate = rates / 15.0
lag = np.array([fl.sampled_position_error(5.0, 1.2, r) for r in rates])
for r, d, e in zip(rates, dose_rate, lag):
    print(f"{r:5.2f} pulses/s: dose rate {d:4.2f} x the 15/s setting, largest lag {e:4.1f} mm")
plt.plot(dose_rate, lag, "o-"); plt.xlabel("relative dose rate"); plt.ylabel("largest lag (mm)"); plt.show()

In [ ]:
# Checks
assert np.all(np.diff(lag) > 0) and np.allclose(dose_rate[1:] / dose_rate[:-1], 0.5)

## Part E · Distance, kerma-area product and staff dose

Air kerma falls with the square of distance from the focal spot while the beam's area grows with the square, so
their product, the kerma-area product, is the same anywhere along the beam (eq-m49-kap). For staff, scattered
radiation from the patient is treated as a point source (eq-m49-staff): dose scales with pulse rate, dose per
pulse and time, falls as $1/r^2$, and halves with each half-value layer of shielding.

In [ ]:
ds = np.linspace(0.5, 1.2, 8)
K = fl.kerma_at(1.0, 0.7, ds); A = fl.field_area_at(100.0, 0.7, ds)
kap = fl.kerma_area_product(K, A)
print("kerma-area product along the beam:", np.round(kap, 6))
plans = {
    "1 m, no shield, 15/s, 10 min": dict(distance_m=1.0, minutes=10, shield_hvl=0, pulse_rate=15),
    "step back to 2 m": dict(distance_m=2.0, minutes=10, shield_hvl=0, pulse_rate=15),
    "2 m and 7.5 pulses/s": dict(distance_m=2.0, minutes=10, shield_hvl=0, pulse_rate=7.5),
    "2 m, 7.5/s, 2 half-value layers": dict(distance_m=2.0, minutes=10, shield_hvl=2, pulse_rate=7.5),
    "... and half the fluoroscopy time": dict(distance_m=2.0, minutes=5, shield_hvl=2, pulse_rate=7.5),
}
rel = {k: float(fl.relative_staff_dose(**v)) for k, v in plans.items()}
for k, v in rel.items():
    print(f"{k:38s} {v:7.4f}")

In [ ]:
# Checks
assert np.allclose(kap, kap[0])
assert abs(rel["... and half the fluoroscopy time"] - 1 / 4 / 2 / 4 / 2) < 1e-12
assert list(rel.values()) == sorted(rel.values(), reverse=True)

Each measure multiplies: together they reduce the dose by a factor of 64 in this example, the kind of
combined reduction a position paper argued for (Heidbuchel et al. 2014). The factors here are illustrative; real
shielding, positions and scatter geometry change them.

## Part F · Reference levels from dose logs

National surveys summarize the kerma-area product of each procedure type. A French survey set reference levels as the
third quartile (and median) of the recorded values (Greffier et al. 2026); a Swiss update took the 75th percentile of
facility medians (Samara et al. 2026). Here, synthetic logs from 30 facilities, with 10 or 30 patients each, show how
the two methods and the number of patients per facility change the answer. All log parameters are illustrative.

In [ ]:
rng = np.random.default_rng(49)
published = {"accessory pathway ablation": 4.10, "persistent AF ablation": 12.65}   # Gy cm^2, Greffier et al. 2026
print("Published third-quartile reference levels (Greffier et al. 2026):", published)
res = {}
for n_pat in (10, 30):
    pooled, medians = [], []
    for rep in range(200):
        logs = fl.synthetic_dose_logs(30, n_pat, median_kap=6.0, spread_between=0.5, spread_within=0.6, rng=rng)
        pooled.append(fl.pooled_reference_level(np.concatenate(logs)))
        medians.append(fl.facility_median_reference_level(logs))
    res[n_pat] = (np.mean(pooled), np.std(pooled), np.mean(medians), np.std(medians))
    print(f"{n_pat:2d} patients per facility: pooled third quartile {res[n_pat][0]:.2f} +/- {res[n_pat][1]:.2f}, "
          f"75th percentile of facility medians {res[n_pat][2]:.2f} +/- {res[n_pat][3]:.2f} (synthetic Gy cm^2)")

In [ ]:
# Checks
assert res[30][1] < res[10][1] and res[30][3] < res[10][3]    # more patients per facility, less variable levels
assert res[30][2] < res[30][0]    # facility medians smooth out within-facility spread, so this level is lower

The method is part of the number: comparing your lab's dose with a reference level only makes sense if both
use the same quantity, the same procedure definition and the same statistic.

## Where to go next

- In part B, add lungs as a low-density region around the heart. How much contrast does the heart's border gain?
- In part C, repeat at 80 keV using the NIST tables. Which contrast changes most, and why?
- In part E, model a shield that covers only part of the body. How would you extend eq-m49-staff?
- In part F, make one facility's doses much higher. Which reference-level method moves more?